# Etapa 3: Aprendizaje No Supervisado (Clustering y Deteccion de Anomalias)
**Materia:** Machine Learning y Ciencia de los Datos (2026)  
**Docente:** Lic. Patricia Rodriguez Bilbao  
**Area Asignada:** Seguridad | **Dataset:** PhiUSIIL Phishing URL Dataset  

---

## Objetivos de la Practica:
1. Cargar las caracteristicas lexico-estructurales escaladas (Z-Score) sin utilizar la etiqueta `label` durante el entrenamiento.
2. Aplicar el algoritmo K-Means: determinar el numero optimo de clusters k mediante el Metodo del Codo (Inertia) y el Coeficiente de Silueta.
3. Aplicar DBSCAN para la identificacion de muestras densas y deteccion de puntos de ruido (URLs anomalas / Zero-Day).
4. Reducir la dimensionalidad mediante PCA (2D) para la proyeccion y visualizacion de clusters.
5. Evaluar la correspondencia entre los clusters descubiertos y las etiquetas reales de phishing.
6. Serializar los modelos en `models/unsupervised/guardados/`.

In [ ]:
import os
import sys

current_dir = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(current_dir, 'requirements.txt')) and os.path.dirname(current_dir) != current_dir:
    current_dir = os.path.dirname(current_dir)
ROOT_DIR = current_dir
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score, homogeneity_score
from src.data_loader import load_phiusiil_sample, load_phiusiil_processed

MODELS_DIR = os.path.join(ROOT_DIR, 'models', 'unsupervised', 'guardados')
RESULTS_DIR = os.path.join(ROOT_DIR, 'models', 'unsupervised', 'resultados')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
print('[OK] Modulos cargados correctamente.')

## 1. Carga y Preparacion de Caracteristicas
Para el analisis de clustering y silueta, utilizaremos una muestra representativa balanceada para optimizar los tiempos de computo.

In [ ]:
# Cargar muestra balanceada de 30,000 URLs
df_sample = load_phiusiil_sample()

TARGET = 'label'
EXCLUDE_COLS = [TARGET, 'URL']
FEATURE_COLS = [col for col in df_sample.select_dtypes(include=[np.number]).columns if col not in EXCLUDE_COLS]

X = df_sample[FEATURE_COLS].values
y_ground_truth = df_sample[TARGET].values

print(f'[INFO] Caracteristicas para Clustering: {X.shape[1]} columnas, {X.shape[0]:,} registros.')

## 2. Metodo del Codo (Elbow Method) y Coeficiente de Silueta
Evaluamos diferentes valores de $k \in [2, 7]$ para determinar el numero optimo de agrupaciones naturales.

In [ ]:
k_values = range(2, 8)
inertias = []
silhouettes = []

# Submuestra rapida para calculo del coeficiente de silueta
idx_sub = np.random.choice(len(X), size=5000, replace=False)
X_sub = X[idx_sub]

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=5)
    km.fit(X)
    inertias.append(km.inertia_)
    
    # Silueta sobre la submuestra
    km_sub = KMeans(n_clusters=k, random_state=42, n_init=5)
    preds_sub = km_sub.fit_predict(X_sub)
    sil = silhouette_score(X_sub, preds_sub)
    silhouettes.append(sil)
    print(f'k = {k} | Inercia: {km.inertia_:.1f} | Silueta: {sil:.4f}')

# Graficar Codo y Silueta
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(k_values, inertias, marker='o', linewidth=2, color='#2980b9')
ax1.set_title('Metodo del Codo (Inercia WCSS)', fontweight='bold')
ax1.set_xlabel('Numero de Clusters (k)')
ax1.set_ylabel('Inercia')

ax2.plot(k_values, silhouettes, marker='s', linewidth=2, color='#27ae60')
ax2.set_title('Coeficiente de Silueta Promedio', fontweight='bold')
ax2.set_xlabel('Numero de Clusters (k)')
ax2.set_ylabel('Score Silueta')

codo_path = os.path.join(RESULTS_DIR, '01_metodo_codo_silueta.png')
plt.savefig(codo_path, dpi=300)
plt.show()

## 3. Modelo Final K-Means y Proyeccion PCA (2D)
Entrenamos el modelo con $k=2$ y proyectamos los clusters usando Analisis de Componentes Principales (PCA).

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X)

# Reduccion PCA a 2 componentes
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
var_exp = pca.explained_variance_ratio_
print(f'[INFO] Varianza explicada por PCA (2D): {var_exp.sum()*100:.2f}% (PC1: {var_exp[0]*100:.1f}%, PC2: {var_exp[1]*100:.1f}%)')

# Visualizacion
plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', alpha=0.5, s=8)
plt.title('Clusters K-Means Proyectados en PCA 2D', fontweight='bold')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.colorbar(scatter, label='Cluster Asignado')
pca_path = os.path.join(RESULTS_DIR, '02_clusters_pca_2d.png')
plt.savefig(pca_path, dpi=300)
plt.show()

## 4. Agrupamiento Basado en Densidad (DBSCAN) y Deteccion de Anomalias
DBSCAN permite aislar URLs anomalas (etiquetadas como ruido: cluster -1), lo que en ciberseguridad representa posibles ataques zero-day.

In [ ]:
# DBSCAN sobre la submuestra para deteccion de outliers
dbscan = DBSCAN(eps=2.5, min_samples=10)
db_labels = dbscan.fit_predict(X_sub)

n_clusters_ = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise_ = list(db_labels).count(-1)

print(f'[INFO] Clusters DBSCAN identificados: {n_clusters_}')
print(f'[INFO] URLs anomalas detectadas (Ruido/Outliers): {n_noise_} ({n_noise_/len(X_sub)*100:.1f}%)')

## 5. Pureza de Agrupamiento y Serializacion

In [ ]:
ari = adjusted_rand_score(y_ground_truth, cluster_labels)
homog = homogeneity_score(y_ground_truth, cluster_labels)

print(f'Indice de Rand Ajustado (ARI): {ari:.4f}')
print(f'Puntaje de Homogeneidad:      {homog:.4f}')

# Guardar modelo K-Means
kmeans_save_path = os.path.join(MODELS_DIR, 'modelo_kmeans.pkl')
joblib.dump(kmeans, kmeans_save_path)
print(f'[OK] Modelo K-Means serializado en: {kmeans_save_path}')